# Generic Uniform B-Splines

- B-spline curves — parametric piece-wise curves determined by control points.
- B-spline functions ­— parametric polinomial real function = factors for corresponding control poins.
- Parameter $t$ — non-decreasing real value along the curve path
- Konts of $t$ — subdivisions of the curve
- A $j$'th Segment of curve corresponds to a span of path with $t \in [t_{i}, t_{i+1})$

In uniform unit-spaced case:

- $t \in [t_{i}, t_{i+1}] \implies i \le t \lt i + 1$
- can be localized within a single segment: $t' = t - t_i \implies => 0.0 \le t' \lt 1.0$
- all B-splines functions are the same over all segments/spans, repeated and phase-shifted


# Generic form

A B-spline function:

$$
B^{(d)}(t) = W^{(d)}(t) · M^{(d)}
$$

> In uniform case doesn't depend on knots placement

A curve segment

$$
S_j^{(d)}(t) = W^{(d)}(t) · M^{(d)} · C^{(d)}_j
$$

> The $C$ is not a real matrix, it's a column-vector of vectors in $\mathbb{R}^2$ or $\mathbb{R}^3$ or anything else in general


### Power vector

The B-splines are polynomial functions of degree $d$.

$$W^{(d)} = [1, t^1, \dots, t^d]$$

> Order of curve is $d-1$ because of how deep it differentiatable (smoothly).

### Control points

Each segment is influenced by $d+1$ nearby control points.

$$C^{(d)}_j = \begin{bmatrix} c_{i} \\ \vdots \\ c_{i+d} \end{bmatrix}$$

> The `nearby` is an arbitrary mapping of indexes of points along the curve.


### The B-Matrices

$$
M^{(1)} =
\begin{pmatrix}
1 & 0 \\
-1 & 1
\end{pmatrix}
$$

$$
M^{(2)} = \frac{1}{2}
\begin{pmatrix}
1 & 1 & 0 \\
-2 & 2 & 0 \\
1 & -2 & 1
\end{pmatrix}
$$

$$
M^{(3)} = \frac{1}{6}
\begin{pmatrix}
1 & 4 & 1 & 0 \\
-3 & 0 & 3 & 0 \\
3 & -6 & 3 & 0 \\
-1 & 3 & -3 & 1
\end{pmatrix}
$$


---


In [ ]:
import numpy as np
from numpy.typing import NDArray
import ipywidgets as wg
import k3d
from k3d.helpers import map_colors

from utils import arr, garr, arrgs, f32

In [ ]:
%%html
<style>
    :root {
        --jp-content-font-color0: var(--vscode-editor-foreground);
        --jp-content-font-color1: var(--vscode-editor-foreground);
        --jp-widgets-color: var(--vscode-editor-foreground);
        --jp-widgets-input-color: var(--vscode-editor-foreground);
        --jp-widgets-input-background-color: var(--vscode-editor-background);
        --jp-widgets-font-size: var(--vscode-editor-font-size);
    }
    .jupyter-widgets input {
        background-color: var(--jp-widgets-input-background-color);
    }
    .cell-output-ipywidget-background {
        background-color: transparent !important;
    }
</style>


#### The $W^{(d)}(t)$

$= \begin{bmatrix} 1 & t & t^2 & \cdots & t^d \end{bmatrix}$

Derivatives:

$\frac{d}{dt} W^{(1)} = 1$

$\frac{d}{dt} W^{(2)} = [1, t] \begin{pmatrix} 1 & 0 \\ 0 & 2 \end{pmatrix}$

$\frac{d}{dt} W^{(3)} = [1, 2t, 3t^2] = [1, t, t^2] \begin{pmatrix} 1 & 0 & 0 \\ 0 & 2 & 0 \\ 0 & 0 & 3\end{pmatrix}$


In [ ]:
def powers(d: int, t: float):
    exps = np.arange(d + 1)
    return t**exps


def np_powers(d: int, t: NDArray):
    exps = np.arange(d + 1)
    return t[..., None] ** exps[None, ...]

#### The $M^{(d)}$


In [ ]:
M_ = {
    1: np.array([[1, 0], [-1, 1]]),
    2: np.array([[1, 1, 0], [-2, 2, 0], [1, -2, 1]]) / 2,
    3: np.array([[1, 4, 1, 0], [-3, 0, 3, 0], [3, -6, 3, 0], [-1, 3, -3, 1]]) / 6,
}

# for derivatives, to multiply with W^(d-1):
Mdt_ = {
    1: np.array([[-1, 1]]),
    2: np.array([[-1, 1, 0], [1, -2, 1]]),
    3: np.array([[-1, 0, 1, 0], [2, -4, 2, 0], [-1, 3, -3, 1]]) / 2,
}

#### The $C_j$

$= \begin{bmatrix}c_1 \\ \vdots \\ c_{d+1}\end{bmatrix}$


In [ ]:
def controls_iter(d: int, controls: NDArray):
    """Iterate over all possible sets of controls"""
    for i in range(len(controls) - d):
        yield controls[i : i + d + 1]

#### The $S^{(d)}_i$

$= W^{(d)} M^{(d)} C^{(d)}_i$


In [ ]:
def point(d: int, controls: NDArray, t: float):
    M = M_[d]
    WM = powers(d, t) @ M
    return WM @ controls


def flow(d: int, controls: NDArray, t: float):
    M = Mdt_[d]
    WM = powers(d - 1, t) @ M
    return WM @ controls


def segm_curve(d: int, controls: NDArray, tspace: NDArray):
    """A segment of curve between the controls,
    with local tspace 0 <= t <= 1"""
    assert len(controls) == d + 1
    M = M_[d]
    WM = np_powers(d, tspace) @ M
    return WM @ controls


def mega_curve(d: int, controls: NDArray, tspace: NDArray):
    """All segments of curve"""
    assert len(controls) > d + 1
    M = M_[d]
    WM = np_powers(d, tspace) @ M
    return np.concat([WM @ ctrl for ctrl in controls_iter(d, controls)])

---


# Plotting


In [ ]:
N = 7
R = 2
D = 16


def random_controls():
    return arr([
        (np.random.random(N) - 0.5) * 2 * R,
        (np.random.random(N) - 0.5) * 2 * R,
        np.arange(0, N),
    ]).T

In [ ]:
controls = random_controls()

In [ ]:
tspace = np.linspace(0.0, 1.0, D, dtype=np.float32)
tspace[-1] -= 0.005

In [ ]:
plot = k3d.Plot(
    height=720,
    background_color=0x404040,
    grid_color=0x383838,
    label_color=0x000000,
    menu_visibility=False,
    grid=(-R, -R, 0.0, R, R, N),
    grid_auto_fit=False,
)
plot.layout = wg.Layout(width="720px", height="720px")

In [ ]:
regenerate_btn = wg.Button(description="regenerate")
curvetype_sel = wg.RadioButtons(description="curve degree", options=[1, 2, 3], value=1)

In [ ]:
wg.HBox([plot, wg.VBox([regenerate_btn, curvetype_sel])], layout=dict(width="100%", grid_gap="8px"))

In [ ]:
k3controls = k3d.line(vertices=[], color=0x808080, line_width=0.125, shader="thick")
k3curve = k3d.line(vertices=[], attribute=[], line_width=0.25, shader="mesh", color_map=k3d.colormaps.basic_color_maps.Rainbow, color_range=[0.0, 1.0])
k3flow = k3d.vectors(origins=[], vectors=[], color=0x808080)

plot += k3controls
plot += k3curve
plot += k3flow

In [ ]:
def rerandomize():
    controls[:] = random_controls()
    k3controls.vertices = f32(controls)
    regenerate()


def regenerate():
    d: int = curvetype_sel.value
    points = mega_curve(d, controls, tspace)
    k3curve.vertices = f32(points)
    k3curve.attribute = np.tile(tspace, len(controls) - d)

    k3flow.origins = garr(point(d, ctrl, 0.5) for ctrl in controls_iter(d, controls))
    k3flow.vectors = garr(flow(d, ctrl, 0.5) for ctrl in controls_iter(d, controls))


rerandomize()

In [ ]:
regenerate_btn.on_click(lambda _: rerandomize())
curvetype_sel.observe(lambda _: regenerate())